In [1]:
# Install the Ultralytics package and check hardware acceleration
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.91 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.0/107.7 GB disk)


In [2]:
# import dependencies
from IPython.display import display, Javascript
from IPython.display import Image as IPythonImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
from PIL import Image
import io
import time

In [3]:
def VideoCapture():
  js = Javascript('''
    async function create(){
      div = document.createElement('div');
      document.body.appendChild(div);

      video = document.createElement('video');
      video.setAttribute('playsinline', '');
      video.width = 400;
      // hide the webcam video
      video.style.display = 'None';
      div.appendChild(video);

      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "environment"}});
      video.srcObject = stream;

      await video.play();

      canvas =  document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      div_out = document.createElement('div');
      document.body.appendChild(div_out);
      img = document.createElement('img');
      img.width = 400;
      div_out.appendChild(img);

    }

    async function capture(){
        return await new Promise(function(resolve, reject){
            pendingResolve = resolve;
            // the width and height of the video
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            // draw the video on the canvas
            canvas.getContext('2d').drawImage(video, 0, 0);
            result = canvas.toDataURL('image/jpeg', 0.7);
            // dropped the quality from 0.8 to 0.7
            pendingResolve(result);
        })
    }

    function showimg(imgb64){
        img.src = "data:image/jpg;base64," + imgb64;
    }

  ''')
  display(js)

def byte2image(byte):
  jpeg = b64decode(byte.split(',')[1])
  im = Image.open(io.BytesIO(jpeg))
  return np.array(im)

def image2byte(image):
  image = Image.fromarray(image)
  buffer = io.BytesIO()
  image.save(buffer, 'jpeg')
  buffer.seek(0)
  x = b64encode(buffer.read()).decode('utf-8')
  return x



In [5]:
from ultralytics import YOLO

VideoCapture()
time.sleep(2)
# Load a model
model = YOLO('yolo26s.pt') # pretraind model
eval_js('create()')
time.sleep(2)



print("Preprocessing live stream...")
try:
  while True:
    byte = eval_js('capture()')
    if not byte:
      print('Stream stopped')
      break
    im = byte2image(byte)
    # Run inference on the live stream
    results = model.predict(
        source = im,
        classes=[0],
        conf=0.45,
        verbose=False
        # add the imgsz to make it small
        #imgsz=320
    )
    # hide labels/confidance
    annotated_frame = results[0].plot(labels=False, conf=False)

    bytes_to_send = image2byte(annotated_frame)
    eval_js(f'showimg("{bytes_to_send}")')

except KeyboardInterrupt:
  print("Stream stopped")


<IPython.core.display.Javascript object>

Preprocessing live stream...
Stream stopped
